In [1]:
from parser import Parser
import shapely
from shapely.ops import transform
import osmnx as ox
import geopandas as gpd
import pathlib
import os


import networkx as nx
import numpy as np
from shapely.geometry import LineString, Point
import math
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt

import numpy.typing as npt

from scipy.spatial import cKDTree
from shapely.strtree import STRtree

from shapely.geometry import LineString
import math

from utils.nx import snap_to_edges, build_final_path, filter_edges

In [2]:
MERCATOR = 3857
WGS_84 = 4326

In [3]:
save_dir = pathlib.Path("/mnt/c/Users/nikita/qgisData/busroutes")
save_dir.mkdir(parents=True, exist_ok=True)

In [4]:
with open("input/boundaries.geojson", "r") as f:
    geojson = f.read()
boundaries = shapely.from_geojson(geojson)
graph = ox.graph_from_polygon(boundaries, network_type="drive", simplify=False)
nodes, edges = ox.graph_to_gdfs(graph)


# convert crs to mercator
nodes = nodes.to_crs(epsg=MERCATOR)
edges = edges.to_crs(epsg=MERCATOR)

geoms = edges["geometry"]
rtree = STRtree(edges["geometry"])

edges["routes"] = 0

In [5]:
# route = "31"
# city = "spb"
# transport_type = "bus"
# route_url = f"/{city}/{transport_type}/{route}"

# _example_route = bus_parser.get_route(route_url)

# # convert to mercator
# example_route_series = gpd.GeoSeries([Point(y, x) for x, y in _example_route], crs=WGS_84).to_crs(epsg=MERCATOR)

In [6]:
bus_parser = Parser.BusGraphParser("Санкт-Петербург")
for route_info in bus_parser.get_all_routes_info():
    route_name = route_info[0]
    route_url = route_info[2]
    print(route_name)
    route_list = bus_parser.get_route(route_url)
    route_series = gpd.GeoSeries([Point(y, x) for x, y in route_list], crs=WGS_84).to_crs(epsg=MERCATOR)

    matched_edges = snap_to_edges(edges, rtree, route_series)
    filtered_edges = filter_edges(edges, matched_edges)
    final_path = build_final_path(graph, filtered_edges)

    pairs = []
    for i in range(len(final_path) - 1):
        u = final_path[i]
        v = final_path[i + 1]
        pairs.append((u, v, 0))
    edges.loc[pairs, "routes"] += 1

read https://kudikina.ru/spb/bus/ from cache
Автобус 1        А.С. "НАЛИЧНАЯ УЛ." - НОВОСИБИРСКАЯ УЛ.
read https://kudikina.ru/spb/bus/1/map from cache
Автобус 1Л        Г. ЛОМОНОСОВ, ВОКЗАЛ - МАЛАЯ ИЖОРА
read https://kudikina.ru/spb/bus/1l/map from cache
Автобус 1М        АВТОБУСНАЯ СТАНЦИЯ "ПР. МАРШАЛА ЖУКОВА" - СТАНЦИЯ МЕТРО "ПРОСПЕКТ ВЕТЕРАНОВ"
read https://kudikina.ru/spb/bus/1m/map from cache
Автобус 1КР        Г. КРОНШТАДТ, ЛЕНИНГРАДСКАЯ ПРИСТАНЬ - МАКАРОВСКИЕ ВОРОТА
read https://kudikina.ru/spb/bus/1kr/map from cache
Автобус 2        А.С. "ПР. МАРШАЛА ЖУКОВА" - СТАНЦИЯ МЕТРО "АДМИРАЛТЕЙСКАЯ"
read https://kudikina.ru/spb/bus/2/map from cache
Автобус 2А        Ж.-Д. СТАНЦИЯ ЛИГОВО - КОМСОМОЛЬСКАЯ ПЛ.
read https://kudikina.ru/spb/bus/2a/map from cache
Автобус 2Л        Г. ЛОМОНОСОВ, ВОКЗАЛ - Г. КРОНШТАДТ, ГРАЖДАНСКАЯ УЛ.
read https://kudikina.ru/spb/bus/2l/map from cache
Автобус 2КР        Г. КРОНШТАДТ, ЛЕНИНГРАДСКАЯ ПРИСТАНЬ - ФОРТ "ШАНЦ"
read https://kudikina.ru/spb/bus/2kr/map 

In [7]:
edges[edges["routes"] > 0]

,,,osmid,highway,junction,lanes,maxspeed,name,oneway,reversed,length,ref,bridge,width,tunnel,access,geometry,routes
u,v,key,,,,,,,,,,,,,,,,
219779,1555933610,0,315569686,primary,circular,5,RU:urban,площадь Победы,True,False,12.762521,NaN,NaN,NaN,NaN,NaN,"LINESTRING (3375261.006 8364793.799, 3375267.2...",22
219780,1555933614,0,142273240,primary,circular,5,RU:urban,площадь Победы,True,False,12.910880,NaN,NaN,NaN,NaN,NaN,"LINESTRING (3375250.954 8364874.412, 3375252.5...",22
219808,389449562,0,201380605,primary,NaN,3,RU:urban,Лиговский проспект,True,False,31.209151,NaN,NaN,NaN,NaN,NaN,"LINESTRING (3376478.396 8376418.668, 3376522.5...",3
219810,2337381944,0,1123339577,primary,NaN,3,RU:urban,Лиговский проспект,True,False,38.258745,NaN,NaN,NaN,NaN,NaN,"LINESTRING (3377104.813 8377247.156, 3377059.5...",4
219811,1832085962,0,33997784,primary_link,NaN,1,RU:urban,NaN,False,False,27.905147,NaN,NaN,NaN,NaN,NaN,"LINESTRING (3377152.291 8377352.477, 3377203.3...",2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12624611898,12624611897,0,1363467184,secondary_link,NaN,1,NaN,NaN,True,False,58.412754,NaN,NaN,NaN,NaN,NaN,"LINESTRING (3308666.637 8403353.967, 3308578.2...",7
12624611899,1460407400,0,40167644,tertiary,NaN,2,RU:urban,Гражданская улица,True,False,1.751069,NaN,NaN,NaN,NaN,NaN,"LINESTRING (3313249.994 8399382.121, 3313253.2...",5
12624611900,1460517028,0,1217529917,residential,NaN,2,RU:urban,Посадская улица,False,False,2.529696,NaN,NaN,NaN,NaN,NaN,"LINESTRING (3313352.341 8399582.668, 3313354.2...",1


In [8]:
# Build a valid path (using shortest paths if needed between edge endpoints)

# if filtered_edges is not None:
# final_path.append(filtered_edges.iloc[-1].name[1])

In [9]:
# final_path = [int(x) for x in final_path]
# final_path[:5]

In [10]:
# for u, v in zip(final_path[:-1], final_path[1:]):
#     # if there are parallel edges, select the shortest in length
#     print(u, v)
#     data = min(graph.get_edge_data(u, v).values(), key=lambda d: d["length"])
#     print(data)

In [11]:
# ox.plot_graph_route(graph, final_path, route_color='red', route_linewidth=2)

In [12]:
edges.to_csv("edges.csv")

In [ ]:
# with open(save_dir / "boundaries.geojson", "w") as f:
#     f.write(shapely.to_geojson(boundaries))

# if "edges.geojson" not in os.listdir(save_dir):
# with open(save_dir / "edges.geojson", "w") as f:
#     f.write(edges.to_json())

# if "nodes.geojson" not in os.listdir(save_dir):
#     with open(save_dir / "nodes.geojson", "w") as f:
#         f.write(nodes.to_json())

# near_points.to_file(save_dir/"points.geojson", driver="GeoJSON")
# edges.to_file(save_dir/"edges.geojson", driver="GeoJSON")
# nodes.to_file(save_dir/"nodes.geojson", driver="GeoJSON")